In [1]:
import requests
from bs4 import BeautifulSoup
import openpyxl
import pandas as pd
import json
from selenium import webdriver
from selenium.webdriver.chrome.service import Service

In [239]:
# Get all review links (review_links.txt)
reviewLinksFile = open("review_links.txt", 'a')

def getReviewsForGivenPage(file, page):
    legacy_url = "https://www.gry-online.pl"
    url = f"https://www.gry-online.pl/recenzje-gier.asp?STR={page}"
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    divs = soup.find_all('div', class_='czyt-lmat-box')
    for d in divs:
        file.write(legacy_url + d.find('a').get('href') + '\n')

def getReviews(file, lastPage):
    for i in range(lastPage):
        getReviewsForGivenPage(file, i + 1)

# getReviews(reviewLinksFile, 137)

reviewLinksFile.close()

In [ ]:
# Clean review links (review_links.txt)
def remove_lines_from_file(input_file, output_file):
    with open(input_file, "r", encoding="utf-8") as file:
        lines = file.readlines()

    # Filtrowanie linii
    filtered_lines = [line for line in lines if '/gramy-dalej.asp' not in line]

    # Zapisanie zmienionych linii do nowego pliku
    with open(output_file, "w", encoding="utf-8") as file:
        file.writelines(filtered_lines)

# remove_lines_from_file("review_links_cleaned.txt", "cleansed_review_links.txt")


In [241]:
def getMonthNumber(monthStr):
    month_variants = {
        "styczeń": 1, "stycznia": 1, "sty": 1,
        "luty": 2, "lutego": 2, "lut": 2,
        "marzec": 3, "marca": 3, "mar": 3,
        "kwiecień": 4, "kwietnia": 4, "kwi": 4,
        "maj": 5, "maja": 5,
        "czerwiec": 6, "czerwca": 6, "cze": 6,
        "lipiec": 7, "lipca": 7, "lip": 7,
        "sierpień": 8, "sierpnia": 8, "sie": 8,
        "wrzesień": 9, "września": 9, "wrz": 9,
        "październik": 10, "października": 10, "paź": 10,
        "listopad": 11, "listopada": 11, "lis": 11,
        "grudzień": 12, "grudnia": 12, "gru": 12
    }

    month_name = monthStr.lower().strip()

    return month_variants.get(month_name, None)

In [ ]:
def getVideoGameInfo(url, driver):
    driver.get(url)

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    section = soup.find('div', class_='S016-game-info')
    try:
        producents = section.find('span', id='game-developer-cnt').find_all('a')
    except:
        producents = []
    try:
        publishers = section.find('span', id='game-publisher-cnt').find_all('a')
    except:
        publishers = []
    try:
        pl_publishers = section.find('span', id='game-publisherpl-cnt').find_all('a')
    except:
        pl_publishers = []
        
    final_dict = {
        'producents': [producent.text for producent in producents],
        'publishers': [publisher.text for publisher in publishers],
        'publishersPL': [publisher.text for publisher in pl_publishers]
    }
    return final_dict

# getVideoGameInfo('https://www.gry-online.pl/gry/the-last-of-us-part-ii-remastered/z56807#ps5')

In [247]:
def getReviewInfo(id, url, driver):
    legacy_url = "https://www.gry-online.pl"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/110.0.0.0 Safari/537.36"
    }

    session = requests.Session()
    session.headers.update(headers)
    
    response = session.get(url)
    
    if response.status_code == 400:
        print("400 Bad Request: The server rejected the request. Try using headers.")
        return None
    elif response.status_code != 200:
        print(f"Request failed with status code {response.status_code}")
        return None
    
    soup = BeautifulSoup(response.text, 'html.parser')

    section = soup.find('section', class_='article-left')
    if not section:
        print("Error: Couldn't find section with class 'article-left'")
        return None
    
    grupa_dane = soup.find('a', class_='grupa-dane-2018-plakie')
    if not grupa_dane:
        print("Error: Couldn't find element with class 'grupa-dane-2018-plakie'")
        return None

    try:
        date = section.find('span', class_='a-d-data').text.split(',')[0].split()
        autor = section.find('p', class_='a-d-aut2-p1').find('span').text
        try:
            score = section.find('div', class_='score').text
        except:
            score = section.find('div', class_='brak-oceny').text
        platform = section.find('p', class_='text-frame-platf2').find('a').text
        game_page_link = legacy_url + section.find('p', class_='text-frame-platf2').find('a').get('href')

        video_game_info = getVideoGameInfo(game_page_link, driver)
        
        genre = grupa_dane.find('p').find('span').text
        title = grupa_dane.find('p').text.replace(" " + genre, '')

        print(f"{title} REVIEW INFO ADDED")
        if id % 50 == 0:
            print("-----------------------------------")
            print(f"CURRENT NUMBER OF REVIEWS: {id}")
            print("-----------------------------------")

        return {
            'id': int(id),
            'title': title,
            'platform': platform,
            'genre': genre,
            'producent': '' if len(video_game_info['producents']) < 1 else video_game_info['producents'][0],
            'producentAlt': '' if len(video_game_info['producents']) < 2 else video_game_info['producents'][1],
            'publisher': '' if len(video_game_info['publishers']) < 1 else video_game_info['publishers'][0],
            'publisherAlt': '' if len(video_game_info['publishers']) < 2 else video_game_info['publishers'][1],
            'publisherPL': '' if len(video_game_info['publishersPL']) < 1 else video_game_info['publishersPL'][0],
            'publisherPLAlt': '' if len(video_game_info['publishersPL']) < 2 else video_game_info['publishersPL'][1],
            'autor': autor,
            'score': score if score == 'brak oceny' else float(score),
            'day': int(date[0]),
            'month': int(getMonthNumber(date[1])),
            'year': int(date[2])
        }
    except AttributeError:
        print("Error: One or more elements were not found in the HTML structure.")
        print(section)
        print(grupa_dane)
        return None


In [244]:
def getInfoForAllReviews(file):
    options = webdriver.ChromeOptions()
    options.add_argument("--headless")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    service = Service()
    driver = webdriver.Chrome(service=service, options=options)

    final_list = []
    i = 1
    for link in file.readlines():
        result = getReviewInfo(i, link.replace('\n', ''), driver)
        if result is not None:
            final_list.append(result)
            i += 1
        else:
            print(f"Error: Something went wrong with {link.replace('\n', '')}")
    
    driver.quit()
    return final_list

In [245]:
reviewLinksFile = open("cleansed_review_links.txt", 'r')

final_data = getInfoForAllReviews(reviewLinksFile)

reviewLinksFile.close()

Like a Dragon: Pirate Yakuza in Hawaii REVIEW INFO ADDED
Avowed REVIEW INFO ADDED
Kingdom Come: Deliverance 2 REVIEW INFO ADDED
Cywilizacja 7 REVIEW INFO ADDED
Donkey Kong Country Returns HD REVIEW INFO ADDED
Indiana Jones i Wielki Krąg REVIEW INFO ADDED
Metaphor: ReFantazio REVIEW INFO ADDED
Microsoft Flight Simulator 2024 REVIEW INFO ADDED
S.T.A.L.K.E.R. 2: Serce Czarnobyla REVIEW INFO ADDED
Farming Simulator 25 REVIEW INFO ADDED
Mario & Luigi: Brothership REVIEW INFO ADDED
Call of Duty: Black Ops 6 REVIEW INFO ADDED
Life is Strange: Double Exposure REVIEW INFO ADDED
Dragon Age: Straż Zasłony REVIEW INFO ADDED
MechWarrior 5: Clans REVIEW INFO ADDED
Super Mario Party Jamboree REVIEW INFO ADDED
Dragon Ball: Sparking! ZERO REVIEW INFO ADDED
Starfield: Shattered Space REVIEW INFO ADDED
Diablo IV: Vessel of Hatred REVIEW INFO ADDED
Silent Hill 2 REVIEW INFO ADDED
EA Sports FC 25 REVIEW INFO ADDED
Ara: History Untold REVIEW INFO ADDED
63 Days REVIEW INFO ADDED
The Legend of Zelda: Echoes o

In [246]:
with open("reviews.json", "w", encoding="utf-8") as file:
    json.dump(final_data, file, ensure_ascii=False, indent=4)

In [19]:
json_file_path = "merged_reviews_cleaned.json"
with open(json_file_path, "r", encoding="utf-8") as file:
    data = json.load(file)

df = pd.DataFrame(data)

excel_file_path = "projectGRYOnlineV2.xlsx"
df.to_excel(excel_file_path, index=False, engine="openpyxl")

print(f"✅ Excel file saved as: {excel_file_path}")


✅ Excel file saved as: projectGRYOnlineV2.xlsx


In [ ]:
# ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
# ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
# ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
# -----------------------------------------------------------------------------------------ERRORS FIXING----------------------------------------------------------------------------------------------
# ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
# ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
# ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
# ERRORS FIXING
def getInfoForAllReviewsWithErrors(links):
    options = webdriver.ChromeOptions()
    options.add_argument("--headless")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    service = Service()
    driver = webdriver.Chrome(service=service, options=options)

    final_list = []
    i = 1
    for link in links:
        result = getReviewInfo(i, link.replace('\n', ''), driver)
        if result is not None:
            final_list.append(result)
            i += 1
        else:
            print(f"Error: Something went wrong with {link.replace('\n', '')}")
    
    driver.quit()
    return final_list

links_with_errors = [
    'https://www.gry-online.pl/recenzje/metaphor-refantazio-recenzja-gry-piekna-basn-na-100-godzin/z246bf',
    'https://www.gry-online.pl/recenzje/recenzja-gry-earthcore-shattered-element-hearthstone-rodem-z-pols/z528ef',
    'https://www.gry-online.pl/recenzje/recenzja-divinity-dragon-commander-strategii-z-elementami-gry-akc/z723aa',
    'https://www.gry-online.pl/recenzje/udany-atak-w-pelnym-slizgu-recenzja-gry-tribes-ascend/za2008',
    'https://www.gry-online.pl/recenzje/doomsday-recenzja-gry/z71231',
    'https://www.gry-online.pl/recenzje/shinobido-way-of-the-ninja-recenzja-gry/zede5',
    'https://www.gry-online.pl/recenzje/ankh-klatwa-mumii-recenzja-gry/zed55',
    'https://www.gry-online.pl/recenzje/spy-hunter-recenzja-gry/z4596',
    'https://www.gry-online.pl/recenzje/strategic-command-european-theater-recenzja-gry/z94da',
    'https://www.gry-online.pl/recenzje/battalia-recenzja-gry/ze338',
    'https://www.gry-online.pl/recenzje/dragon-court-recenzja-gry/z51bb'
]

r = getInfoForAllReviewsWithErrors(links_with_errors)

with open("reviews_with_errors.json", "w", encoding="utf-8") as file:
    json.dump(r, file, ensure_ascii=False, indent=4)

In [ ]:
# ERROR FIXING
with open("reviews.json", "r", encoding="utf-8") as file1:
    reviews1 = json.load(file1)

with open("reviews_with_errors.json", "r", encoding="utf-8") as file2:
    reviews2 = json.load(file2)

# Merge both lists
merged_reviews = reviews1 + reviews2

# Remove duplicates by creating a set of unique reviews (ignoring the 'id' field)
unique_reviews = []
seen = set()

for review in merged_reviews:
    # Create a unique identifier excluding the 'id' field
    review_copy = {k: v for k, v in review.items() if k != "id"}
    review_tuple = tuple(sorted(review_copy.items()))  # Convert to a hashable format

    if review_tuple not in seen:
        seen.add(review_tuple)
        unique_reviews.append(review)
    else:
        print(review_tuple)

# Sort reviews by date (oldest first)
unique_reviews.sort(key=lambda x: (x["year"], x["month"], x["day"]))

# Reassign IDs starting from 1
for new_id, review in enumerate(unique_reviews, start=1):
    review["id"] = new_id

# Save the cleaned list to a new JSON file
output_path = "merged_reviews_cleaned.json"
with open(output_path, "w", encoding="utf-8") as output_file:
    json.dump(unique_reviews, output_file, indent=4, ensure_ascii=False)

print(f"✅ Merged, deduplicated, and sorted reviews saved to: {output_path}")



In [ ]:
# ERROR FIXING
with open("merged_reviews_cleaned.json", "r", encoding="utf-8") as file1:
    reviews = json.load(file1)

# Reassign IDs starting from 1
for new_id, review in enumerate(reviews, start=1):
    review["id"] = new_id

# Save the cleaned list to a new JSON file
output_path = "merged_reviews_cleaned_v2.json"
with open(output_path, "w", encoding="utf-8") as output_file:
    json.dump(reviews, output_file, indent=4, ensure_ascii=False)

print(f"✅ Merged, deduplicated, and sorted reviews saved to: {output_path}")



In [ ]:
# ERROR FIXING
# Define file paths
input_file = "review_links.txt"
output_file = "review_links_cleaned.txt"

# Read all lines from the file
with open(input_file, "r", encoding="utf-8") as file:
    lines = file.readlines()

# Remove duplicates while preserving order
unique_links = list(dict.fromkeys(line.strip() for line in lines))

# Save the cleaned list to a new file
with open(output_file, "w", encoding="utf-8") as file:
    file.write("\n".join(unique_links))

print(f"✅ Cleaned file saved to: {output_file}")
